## 4. EfficientDet

This section trains and tunes **EfficientDet**, the third detector in the comparison, on exactly the same 427-image dataset and the same Train/Validation/Test split used for YOLO (Section 1.1).

**Why EfficientDet as the third model?** The study compares three different design philosophies rather than three variants of the same idea:

| Model | Family | Key idea |
|---|---|---|
| YOLOv8 | One-stage | Anchor-free head, very fast, optimised for real-time use |
| Faster R-CNN | Two-stage | Region proposals first, then classification; usually accurate but slower |
| **EfficientDet** | One-stage | EfficientNet backbone, **BiFPN** multi-scale feature fusion, **compound scaling** (D0 to D7), focal loss |

EfficientDet is also one-stage like YOLO, but it is built on a different backbone, a different feature-fusion neck and an anchor-based head. Comparing it with YOLO therefore shows whether the differences we observe come from the one-stage/two-stage distinction or from the specific architecture.

EfficientDet relies on a BiFPN architecture. Following the team protocol, it is initialised randomly, ensuring no pre-trained backbone is loaded.

**Responsibilities of this section:** (i) training EfficientDet, (ii) hyperparameter optimisation on the validation set, (iii) providing the loss curves and validation mAP for Section 5, and the test predictions in a common format for Sections 6–8.

### 4.1 Related work (EfficientDet)

- **Architecture.** EfficientDet [1] combines the EfficientNet backbone [2] with a weighted bidirectional feature pyramid (BiFPN), an extension of FPN [3]. Resolution, depth and width are scaled together with a single compound coefficient, giving the D0 to D7 family. Classification is trained with the focal loss [4], which down-weights easy background anchors.
- **Speed and accuracy trade-off.** Huang et al. [5] showed that detector comparisons are only meaningful when the backbone, input resolution and hardware are reported, because these factors change speed and accuracy as much as the meta-architecture does. We therefore report all of them (Section 4.3).
- **Agricultural use.** EfficientDet and SSD were compared for strawberry disease detection on embedded hardware [6]. Faster R-CNN variants have been used for strawberry detection and plumpness grading [7]. YOLOv8n has been applied to strawberry quality classification, and the authors explicitly list a comparison with Faster R-CNN and EfficientDet as future work [8]. This is the gap our study addresses.
- **Data augmentation** is the standard remedy for small datasets [9]. It must however preserve the label: our classes are defined by **physical length**, so transformations that change the apparent size of the fruit (scaling, zoom crops, mosaic) may contradict the label. We test this explicitly (Experiment E3).
- **Evaluation and leakage.** Different toolkits compute mAP differently [10], so all three models are scored with the same COCO protocol [11]. Data leakage between training and test sets is a common cause of over-optimistic results in ML studies [12], so the split is created once, before any augmentation, and the test set is used only once.
- **Hyperparameter search.** Under a limited compute budget, a staged search (one factor at a time from a baseline) is used instead of a full grid. Random/Bayesian search [13] is listed as future work.

References are listed in `docs/literature_review.md`.

### 4.2 Experimental protocol

1. **Same split as YOLO.** The split folders created in Section 1.1 (seed = 42, 341/43/43 images) are reused, not re-created. All models therefore see the same images.
2. **Aspect-ratio-preserving resize (letterbox).** Images are resized so that the longest side equals the model input size, then padded with grey (114). Stretching the image would distort the length and width of the fruit, and our classes depend on length.
3. **Training from scratch (team policy).** As for YOLO (Section 2) and Faster R-CNN (Section 3), EfficientDet is initialised with **random weights**: no COCO detector weights and no ImageNet backbone (`PRETRAINED = False`, `pretrained_backbone = False`). All three models therefore start from the same point and the comparison measures the architectures, not their pretraining. Every run uses at most 100 epochs with early stopping (patience 20 epochs on validation mAP50-95).
4. **Model selection on validation only.** Every experiment is ranked by validation mAP50-95. The test set is evaluated **once**, with the final configuration (Section 4.6).
5. **Common metrics.** mAP50 and mAP50-95 use the COCO protocol (pycocotools via `torchmetrics`). Precision, Recall, F1 and the confusion matrix are computed at IoU = 0.5 and a fixed confidence threshold. Inference time is measured on the GPU at batch size 1.
6. **Everything is logged.** Each run appends one row to `experiment_log.csv`, including failed or discarded runs, so that the full process can be reported.

### 4.3 Setup (local run)

This version runs on a local computer with an NVIDIA GPU instead of Google Colab. The model code, the split, the hyperparameters and the metrics are identical to the Colab notebook; only the file paths and the data loading change:

- the images are read directly from the labelled folder (no copy into `yolo_dataset`)
- the split is taken from `splits.json`, the file saved by the Colab split cell, so every model uses the same images
- the data pipeline lives in `effdet_lib.py`, so that several CPU cores can load images in parallel (on Windows, worker processes cannot use classes defined inside a notebook)
- 4 worker processes load training images and 2 load validation images. With 8 + 8 workers the first local run ran out of system RAM, because on Windows every worker process loads its own copy of PyTorch

`effdet_lib.py` and `splits.json` must be in the same folder as this notebook.

In [ ]:
import os, json, time, math, random, functools, platform
import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
import albumentations as A
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torchvision.ops import box_iou
from torchmetrics.detection import MeanAveragePrecision
from sklearn.model_selection import train_test_split
import effdet
from effdet import create_model
from effdet_lib import (MEAN, STD, read_image, read_yolo_labels, build_augment, letterbox,
                        StrawberryDataset, collate, worker_init)

DATA_SRC = r'E:\Çilek Boy Numaralanmış'
SPLITS_JSON = 'splits.json'
OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

CLASS_NAMES = ['Short', 'Medium', 'Long']
NUM_CLASSES = len(CLASS_NAMES)
SEED = 42
PRETRAINED = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS = min(4, os.cpu_count() or 2)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

env = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'torchvision': torchvision.__version__, 'albumentations': A.__version__,
    'device': DEVICE, 'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none',
    'cpu_workers': NUM_WORKERS,
}
print(json.dumps(env, indent=2))
json.dump(env, open(f'{OUT_DIR}/environment.json', 'w'), indent=2)
assert DEVICE == 'cuda', 'GPU not found: install the CUDA build of PyTorch (see the setup guide).'
assert os.path.isdir(DATA_SRC), f'Data folder not found: {DATA_SRC}'

### 4.4 Data pipeline

The split is read from `splits.json` (created by the Colab split cell from the sorted file list, `random_state = 42`). If the file is missing, the same split is recomputed with the same procedure. The class distribution and the leakage check are printed as in the Colab notebook; the numbers must match (train / val / test = 341 / 43 / 43).

In [ ]:
SPLITS = ['train', 'val', 'test']
IMG_EXT = ('.jpg', '.jpeg', '.png')

images = sorted(f for f in os.listdir(DATA_SRC) if f.lower().endswith(IMG_EXT))
ext_of = {os.path.splitext(f)[0]: f for f in images}
stems = sorted(ext_of)

if os.path.exists(SPLITS_JSON):
    split_names = json.load(open(SPLITS_JSON, encoding='utf-8'))
    print('Split loaded from', SPLITS_JSON)
else:
    tr, tmp = train_test_split(stems, test_size=0.2, random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.5, random_state=SEED)
    split_names = {'train': sorted(tr), 'val': sorted(va), 'test': sorted(te)}
    json.dump(split_names, open(SPLITS_JSON, 'w', encoding='utf-8'), indent=1)
    print('splits.json not found: split recomputed and saved')

missing = [s for v in split_names.values() for s in v if s not in ext_of]
assert not missing, f'Images listed in splits.json but not found in DATA_SRC: {missing[:5]}'

FILES = {s: [(os.path.join(DATA_SRC, ext_of[n]), os.path.join(DATA_SRC, n + '.txt')) for n in split_names[s]]
         for s in SPLITS}

dist = pd.DataFrame(0, index=CLASS_NAMES, columns=SPLITS)
for s in SPLITS:
    for _, lbl in FILES[s]:
        for r in read_yolo_labels(lbl):
            dist.loc[CLASS_NAMES[r[0]], s] += 1
dist.loc['images'] = [len(FILES[s]) for s in SPLITS]
print(dist)
dist.to_csv(f'{OUT_DIR}/class_distribution.csv')

names = {s: set(split_names[s]) for s in SPLITS}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not (names[a] & names[b]), f'Same file in {a} and {b}'
print('Leakage check (file names): OK')

In [ ]:
def make_loader(split, img_size, batch_size, augment_mode='none', shuffle=False):
    aug = build_augment(augment_mode) if split == 'train' else None
    ds = StrawberryDataset(FILES[split], img_size, aug)
    workers = NUM_WORKERS if split == 'train' else min(2, NUM_WORKERS)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=workers,
                      persistent_workers=workers > 0, pin_memory=True, worker_init_fn=worker_init,
                      collate_fn=functools.partial(collate, img_size=img_size), drop_last=False)

def show_batch(augment_mode='flip', n=6, img_size=512):
    ds = StrawberryDataset(FILES['train'], img_size, build_augment(augment_mode))
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax, i in zip(axes, random.sample(range(len(ds)), n)):
        x, t, _ = ds[i]
        img = np.clip((x.numpy().transpose(1, 2, 0) * STD + MEAN), 0, 1)
        ax.imshow(img); ax.axis('off')
        for (x1, y1, x2, y2), c in zip(t['bbox_xyxy'].tolist(), t['labels'].tolist()):
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', lw=1.5))
            ax.text(x1, y1, CLASS_NAMES[c], color='lime', fontsize=8)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/sanity_check_{augment_mode}.png', dpi=120); plt.show()

show_batch('flip')

In [ ]:
# Speed check: time to load one epoch of training images with the parallel workers
t0 = time.time()
for _ in make_loader('train', 512, 8, 'flip', shuffle=True):
    pass
print(f'One epoch of data loading: {time.time() - t0:.1f} s with {NUM_WORKERS} workers')

### 4.5 Training and evaluation functions

- **Optimiser:** AdamW with linear warm-up followed by a cosine learning-rate decay.
- **Early stopping:** training stops when validation mAP50-95 has not improved for `patience` epochs, and the best checkpoint is kept. Cordeiro et al. [8] observed rising validation loss in long YOLO runs and adopted early stopping for the same reason.
- **Metrics:**
  - mAP50 and mAP50-95 follow the COCO protocol, with a score threshold of 0.001 so that the full precision-recall curve is used.
  - Precision, Recall and F1 are computed at IoU ≥ 0.5 and confidence ≥ `CONF_THR`. Each prediction is matched to at most one ground-truth box. A box at the right place with the wrong class counts as one FP and one FN, and appears off-diagonal in the confusion matrix. The last row and column of the matrix are *background* (missed fruit / false alarms).

In [ ]:
CONF_THR = 0.25      # confidence threshold for P/R/F1 and the confusion matrix (same value for all models)
IOU_THR = 0.5

@torch.no_grad()
def predict(model, loader, score_thr=0.001):
    # Returns per-image predictions and ground truth (xyxy, model input space, 0-based classes) and the mean validation loss.
    model.eval()
    preds, gts, metas, losses = [], [], [], []
    for imgs, et, raw, meta in loader:
        imgs = imgs.to(DEVICE)
        et = {k: v.to(DEVICE) for k, v in et.items()}
        out = model(imgs, et)
        losses.append(out['loss'].item())
        for det, t, m in zip(out['detections'].cpu(), raw, meta):
            keep = det[:, 4] >= score_thr
            det = det[keep]
            preds.append({'boxes': det[:, :4], 'scores': det[:, 4], 'labels': det[:, 5].long() - 1})
            gts.append({'boxes': t['bbox_xyxy'], 'labels': t['labels']})
            metas.append(m)
    return preds, gts, metas, float(np.mean(losses))

def coco_map(preds, gts):
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=True)
    metric.update(preds, gts)
    r = metric.compute()
    out = {'mAP50': r['map_50'].item(), 'mAP50-95': r['map'].item()}
    per_cls = {}
    classes = r['classes'].tolist() if r['classes'].ndim else [r['classes'].item()]
    for c, v in zip(classes, r['map_per_class'].tolist() if r['map_per_class'].ndim else [r['map_per_class'].item()]):
        per_cls[CLASS_NAMES[c]] = v
    out['mAP50-95_per_class'] = per_cls
    return out

def match_counts(preds, gts, conf_thr=CONF_THR, iou_thr=IOU_THR):
    # Greedy one-to-one matching by IoU. Returns per-class TP/FP/FN and a confusion matrix.
    # Confusion matrix: rows = true class, columns = predicted class, index C = background.
    C = NUM_CLASSES
    cm = np.zeros((C + 1, C + 1), dtype=int)
    tp, fp, fn = np.zeros(C), np.zeros(C), np.zeros(C)
    for p, g in zip(preds, gts):
        k = p['scores'] >= conf_thr
        pb, pl = p['boxes'][k], p['labels'][k]
        gb, gl = g['boxes'], g['labels']
        used_p, used_g = set(), set()
        if len(pb) and len(gb):
            iou = box_iou(gb, pb)
            pairs = [(iou[i, j].item(), i, j) for i in range(len(gb)) for j in range(len(pb)) if iou[i, j] >= iou_thr]
            for _, i, j in sorted(pairs, reverse=True):
                if i in used_g or j in used_p:
                    continue
                used_g.add(i); used_p.add(j)
                t, q = gl[i].item(), pl[j].item()
                cm[t, q] += 1
                if t == q:
                    tp[t] += 1
                else:
                    fp[q] += 1; fn[t] += 1
        for j in range(len(pb)):
            if j not in used_p:
                cm[C, pl[j].item()] += 1; fp[pl[j].item()] += 1
        for i in range(len(gb)):
            if i not in used_g:
                cm[gl[i].item(), C] += 1; fn[gl[i].item()] += 1
    return tp, fp, fn, cm

def prf(tp, fp, fn):
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    r = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    return p, r, f1

def evaluate(model, loader):
    preds, gts, metas, loss = predict(model, loader)
    m = coco_map(preds, gts)
    tp, fp, fn, cm = match_counts(preds, gts)
    p, r, f1 = prf(tp, fp, fn)
    m.update({'loss': loss, 'precision': p.mean(), 'recall': r.mean(), 'f1': f1.mean(),
              'per_class_prf': {CLASS_NAMES[c]: (p[c], r[c], f1[c]) for c in range(NUM_CLASSES)}})
    return m, (preds, gts, metas, cm)

In [ ]:
VARIANT_SIZE = {'tf_efficientdet_d0': 512, 'tf_efficientdet_d1': 640, 'tf_efficientdet_d2': 768}

BASE_CFG = {
    'id': 'E0', 'note': 'baseline',
    'variant': 'tf_efficientdet_d0', 'img_size': None,      # None -> native size of the variant
    'pretrained': PRETRAINED,          # False: no COCO detector weights
    'pretrained_backbone': False,      # False: no ImageNet backbone either -> fully random init
    'freeze_backbone': False,
    'augment': 'flip',
    'lr': 1e-3, 'weight_decay': 1e-4, 'batch_size': 8,
    'epochs': 100, 'warmup_epochs': 5, 'patience': 20,
    'seed': SEED,
}

def build_model(cfg):
    size = cfg['img_size'] or VARIANT_SIZE[cfg['variant']]
    model = create_model(cfg['variant'], bench_task='train', bench_labeler=True,
                         num_classes=NUM_CLASSES, pretrained=cfg['pretrained'],
                         pretrained_backbone=cfg['pretrained_backbone'], image_size=(size, size))
    if cfg['freeze_backbone']:
        for p in model.model.backbone.parameters():
            p.requires_grad = False
    return model.to(DEVICE), size

def train_experiment(cfg, verbose=True):
    cfg = {**BASE_CFG, **cfg}
    set_seed(cfg['seed'])
    model, size = build_model(cfg)
    train_loader = make_loader('train', size, cfg['batch_size'], cfg['augment'], shuffle=True)
    val_loader = make_loader('val', size, cfg['batch_size'])

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    steps_per_epoch = len(train_loader)
    total, warm = cfg['epochs'] * steps_per_epoch, cfg['warmup_epochs'] * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    use_amp = DEVICE == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    run_dir = f"{OUT_DIR}/runs/{cfg['id']}"
    os.makedirs(run_dir, exist_ok=True)
    history, best, best_epoch, bad = [], -1.0, -1, 0
    t0 = time.time()
    for epoch in range(cfg['epochs']):
        model.train()
        tr = {'loss': 0., 'class_loss': 0., 'box_loss': 0.}
        for imgs, et, _, _ in train_loader:
            imgs = imgs.to(DEVICE)
            et = {k: v.to(DEVICE) for k, v in et.items()}
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                out = model(imgs, et)
            opt.zero_grad(set_to_none=True)
            scaler.scale(out['loss']).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 10.0)
            scaler.step(opt); scaler.update(); sched.step()
            for k in tr:
                tr[k] += out[k].item() / steps_per_epoch
        val, _ = evaluate(model, val_loader)
        row = {'epoch': epoch + 1, 'train_loss': tr['loss'], 'train_cls_loss': tr['class_loss'],
               'train_box_loss': tr['box_loss'], 'val_loss': val['loss'], 'val_mAP50': val['mAP50'],
               'val_mAP50-95': val['mAP50-95'], 'val_f1': val['f1'], 'lr': opt.param_groups[0]['lr']}
        history.append(row)
        if verbose:
            print(f"[{cfg['id']}] ep {epoch+1:3d} train {tr['loss']:.4f} | val loss {val['loss']:.4f} "
                  f"mAP50 {val['mAP50']:.3f} mAP50-95 {val['mAP50-95']:.3f} F1 {val['f1']:.3f}")
        if val['mAP50-95'] > best:
            best, best_epoch, bad = val['mAP50-95'], epoch + 1, 0
            torch.save(model.state_dict(), f'{run_dir}/best.pt')
        else:
            bad += 1
            if bad >= cfg['patience']:
                if verbose: print(f'Early stopping at epoch {epoch+1} (best epoch {best_epoch})')
                break
    train_time = time.time() - t0
    pd.DataFrame(history).to_csv(f'{run_dir}/history.csv', index=False)
    json.dump(cfg, open(f'{run_dir}/config.json', 'w'), indent=2)

    model.load_state_dict(torch.load(f'{run_dir}/best.pt', map_location=DEVICE))
    val, _ = evaluate(model, val_loader)
    result = {'id': cfg['id'], 'note': cfg['note'], 'variant': cfg['variant'], 'img_size': size,
              'pretrained': cfg['pretrained'], 'freeze_backbone': cfg['freeze_backbone'],
              'augment': cfg['augment'], 'lr': cfg['lr'], 'batch_size': cfg['batch_size'],
              'best_epoch': best_epoch, 'epochs_run': len(history), 'train_time_min': round(train_time / 60, 1),
              'val_mAP50': round(val['mAP50'], 4), 'val_mAP50-95': round(val['mAP50-95'], 4),
              'val_P': round(val['precision'], 4), 'val_R': round(val['recall'], 4), 'val_F1': round(val['f1'], 4),
              'timestamp': time.strftime('%Y-%m-%d %H:%M')}
    log_path = f'{OUT_DIR}/experiment_log.csv'
    pd.DataFrame([result]).to_csv(log_path, mode='a', header=not os.path.exists(log_path), index=False)
    return result, model, history

In [ ]:
def plot_history(exp_id):
    h = pd.read_csv(f'{OUT_DIR}/runs/{exp_id}/history.csv')
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='val')
    ax[0].set_title('Total loss (focal + box)'); ax[0].legend()
    ax[1].plot(h.epoch, h.train_cls_loss, label='class'); ax[1].plot(h.epoch, h.train_box_loss, label='box')
    ax[1].set_title('Training loss components'); ax[1].legend()
    ax[2].plot(h.epoch, h['val_mAP50'], label='mAP50'); ax[2].plot(h.epoch, h['val_mAP50-95'], label='mAP50-95')
    ax[2].set_title('Validation mAP'); ax[2].legend()
    for a in ax: a.set_xlabel('epoch')
    plt.suptitle(exp_id); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/runs/{exp_id}/curves.png', dpi=150); plt.show()

def show_log():
    return pd.read_csv(f'{OUT_DIR}/experiment_log.csv')

### 4.6 Hyperparameter optimisation

A full grid over every hyperparameter is not affordable on a free GPU: one run takes 30–90 minutes, and with 43 validation images small differences are within noise. We therefore use a **staged search**. Each stage changes one factor, keeps the best value found so far and tests a hypothesis stated in advance.

| Stage | Factor | Values | Hypothesis / reason |
|---|---|---|---|
| E0 | Baseline | D0, 512 px, random init, lr 1e-3, `flip` aug | Reference point |
| E1 | Learning rate | 3e-4, 1e-3 (E0), 3e-3 | Random initialisation needs a larger lr than fine-tuning, but focal-loss training can diverge if it is too high |
| E3 | Augmentation | `flip` vs. `flip` + random scale (0.7–1.3) | **Size-label consistency:** scaling changes the apparent size of the fruit and is expected to *hurt* Short/Medium/Long separation |
| FINAL | Seed | 42, 7, 2024 | The final configuration is trained with three seeds; mean ± standard deviation is reported |

**Selection rule:** highest validation mAP50-95. If two runs differ by less than 0.01, the simpler one is preferred, because a difference that small on 43 images is not meaningful.

Runs that fail, crash or are interrupted are **kept in the log** and explained in Section 4.8.

E0 was trained on Google Colab (Tesla T4) before switching to the local GPU; its result is in the experiment log. Running it again here is optional (it shows the effect of the hardware on training time only).

In [ ]:
# E0 - baseline (random initialisation)
res, model, _ = train_experiment({'id': 'E0', 'note': 'baseline, from scratch'})
print(res); plot_history('E0')

In [ ]:
# E1 - learning rate
for i, lr in enumerate([3e-4, 3e-3]):            # 1e-3 is already E0
    res, _, _ = train_experiment({'id': f'E1_{i}', 'note': f'lr={lr}', 'lr': lr})
    print(res)
show_log()

In [ ]:
BEST = {'lr': 3e-3}                               # selected in E1 (E1_1)

In [ ]:
# E3 - augmentation: does size-altering augmentation hurt length-based classes?
res, _, _ = train_experiment({**BEST, 'id': 'E3_scale', 'note': 'augment=scale', 'augment': 'scale'})
print(res); plot_history('E3_scale')

In [ ]:
BEST.update({'augment': 'flip', 'variant': 'tf_efficientdet_d0', 'batch_size': 8})   # E3: scaling rejected
json.dump(BEST, open(f'{OUT_DIR}/best_config.json', 'w'), indent=2)

# Seed robustness of the final configuration (seed 42 is E1_1)
seed_runs = []
for s in [7, 2024]:
    res, _, _ = train_experiment({**BEST, 'id': f'FINAL_s{s}', 'note': f'final config, seed {s}', 'seed': s})
    seed_runs.append(res)
    print(res)
log = show_log()
sr = log[log['id'].isin(['E1_1', 'FINAL_s7', 'FINAL_s2024'])]
print(sr[['id', 'val_mAP50', 'val_mAP50-95', 'val_F1']])
print('mean ± std  mAP50-95:', f"{sr['val_mAP50-95'].mean():.4f} ± {sr['val_mAP50-95'].std():.4f}")

### 4.7 Test-set inference for Sections 6–8 (run once)

The test set is used **only here**, with the configuration chosen on the validation set. Running it again after changing settings would turn the test set into a second validation set and bias the comparison with YOLO and Faster R-CNN.

This step produces:
- the test metrics table (overall and per class)
- the confusion matrix and the F1-confidence curve (same plots as the YOLO section, for direct comparison)
- the inference time (ms/image and FPS), measured on the GPU at batch size 1
- `effdet_test_predictions.json`: COCO-format detections in **original image pixels**, for the joint evaluation in Section 5

In [ ]:
FINAL_ID = 'E1_1'                                 # final configuration, seed 42
cfg = {**BASE_CFG, **json.load(open(f'{OUT_DIR}/runs/{FINAL_ID}/config.json'))}
model, size = build_model({**cfg, 'pretrained': False, 'pretrained_backbone': False})   # weights come from the checkpoint
model.load_state_dict(torch.load(f'{OUT_DIR}/runs/{FINAL_ID}/best.pt', map_location=DEVICE))
test_loader = make_loader('test', size, batch_size=1)

test, (preds, gts, metas, cm) = evaluate(model, test_loader)
rows = [{'class': 'all', 'P': test['precision'], 'R': test['recall'], 'F1': test['f1'],
         'mAP50': test['mAP50'], 'mAP50-95': test['mAP50-95']}]
for c in CLASS_NAMES:
    p, r, f1 = test['per_class_prf'][c]
    rows.append({'class': c, 'P': p, 'R': r, 'F1': f1, 'mAP50': np.nan,
                 'mAP50-95': test['mAP50-95_per_class'].get(c, np.nan)})
test_table = pd.DataFrame(rows).round(4)
test_table.to_csv(f'{OUT_DIR}/test_metrics.csv', index=False)
test_table

In [ ]:
# Confusion matrix (rows: true, columns: predicted, last = background)
labels = CLASS_NAMES + ['background']
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'EfficientDet confusion matrix (test, conf ≥ {CONF_THR}, IoU ≥ {IOU_THR})')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix_test.png', dpi=150); plt.show()

# F1-confidence curve (per class and overall)
ths = np.linspace(0.05, 0.95, 19)
f1s = np.array([prf(*match_counts(preds, gts, conf_thr=t)[:3])[2] for t in ths])
plt.figure(figsize=(6, 4))
for c in range(NUM_CLASSES):
    plt.plot(ths, f1s[:, c], label=CLASS_NAMES[c])
plt.plot(ths, f1s.mean(1), 'k', lw=2, label=f'all (max {f1s.mean(1).max():.2f} at {ths[f1s.mean(1).argmax()]:.2f})')
plt.xlabel('Confidence'); plt.ylabel('F1'); plt.legend(); plt.title('EfficientDet F1-confidence (test)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/f1_confidence_test.png', dpi=150); plt.show()

In [ ]:
# Inference time: model forward + post-processing (NMS), batch size 1, excluding image loading.
# Measure every model on the same GPU and the same input size, otherwise the speed comparison is not valid.
@torch.no_grad()
def measure_speed(model, size, n_warmup=20, n_runs=200):
    model.eval()
    x = torch.randn(1, 3, size, size, device=DEVICE)
    info = {'img_scale': torch.ones(1, device=DEVICE),
            'img_size': torch.tensor([[size, size]], dtype=torch.float32, device=DEVICE)}
    predictor = effdet.DetBenchPredict(model.model).to(DEVICE).eval()
    for _ in range(n_warmup):
        predictor(x, info)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_runs):
        predictor(x, info)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / n_runs * 1000
    return {'ms_per_image': round(ms, 2), 'fps': round(1000 / ms, 1), 'input_size': size,
            'params_M': round(sum(p.numel() for p in model.model.parameters()) / 1e6, 2), 'device': env['gpu']}

speed = measure_speed(model, size)
print(speed)
json.dump({**{k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in test.items()
              if k in ('mAP50', 'mAP50-95', 'precision', 'recall', 'f1')}, **speed},
          open(f'{OUT_DIR}/test_summary.json', 'w'), indent=2)

In [ ]:
# Export test predictions in COCO format, in ORIGINAL image pixel coordinates (letterbox inverted),
# so the evaluation member can score all three models with the same script.
coco_preds = []
for p, m in zip(preds, metas):
    left, top = m['pad']; r = m['scale']
    for (x1, y1, x2, y2), s, c in zip(p['boxes'].tolist(), p['scores'].tolist(), p['labels'].tolist()):
        x1, x2 = (x1 - left) / r, (x2 - left) / r
        y1, y2 = (y1 - top) / r, (y2 - top) / r
        coco_preds.append({'file_name': os.path.basename(m['path']), 'category_id': int(c),
                           'category_name': CLASS_NAMES[int(c)],
                           'bbox': [round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],   # x, y, w, h
                           'score': round(s, 5)})
json.dump(coco_preds, open(f'{OUT_DIR}/effdet_test_predictions.json', 'w'))
print(len(coco_preds), 'detections exported ->', f'{OUT_DIR}/effdet_test_predictions.json')

In [ ]:
# Qualitative results: predictions (red) vs ground truth (green) on test images
def show_predictions(n=6, conf=CONF_THR):
    idx = random.sample(range(len(preds)), min(n, len(preds)))
    fig, axes = plt.subplots(1, len(idx), figsize=(3.5 * len(idx), 3.5))
    for ax, i in zip(np.atleast_1d(axes), idx):
        img, _, _, _ = letterbox(read_image(metas[i]['path']), size)
        ax.imshow(img); ax.axis('off')
        for (x1, y1, x2, y2), c in zip(gts[i]['boxes'].tolist(), gts[i]['labels'].tolist()):
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', lw=1.5))
        k = preds[i]['scores'] >= conf
        for (x1, y1, x2, y2), c, s in zip(preds[i]['boxes'][k].tolist(), preds[i]['labels'][k].tolist(), preds[i]['scores'][k].tolist()):
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='red', lw=1.5))
            ax.text(x1, y2 + 12, f'{CLASS_NAMES[c]} {s:.2f}', color='red', fontsize=8)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/qualitative_test.png', dpi=150); plt.show()

show_predictions()

### 4.8 Experiment log and process notes

All runs, including interrupted ones. Full notes are in `docs/experiment_log.md`.

| ID | Change vs. previous best | Hypothesis | val mAP50 | val mAP50-95 | Decision |
|---|---|---|---|---|---|
| E0 | baseline (D0, 512, random init, lr 1e-3, flip) | reference | 0.974 | 0.913 | reference |
| E1_0 | lr = 3e-4 | lower lr may under-fit from scratch | 0.961 | 0.862 | rejected |
| E1_1 | lr = 3e-3 | higher lr converges faster but may diverge | 0.983 | 0.954 | **selected** |
| E3_scale | + random scale 0.7–1.3 | expected to hurt size classes | 0.963 | 0.925 (interrupted at epoch 76) | rejected, keep `flip` |
| FINAL | seeds 42 / 7 / 2024 | variance across seeds | 0.977 ± 0.005 | 0.940 ± 0.018 | reported result |

**Process notes** (*what we tried, what happened, why, what we changed*):
- **Split.** The original split listed files with `os.listdir`, whose order is not guaranteed, so the same seed gave different sets on different machines. The file list is now sorted and the split is saved as `splits.json` for all models.
- **Speed.** On the free Colab T4 one epoch took ~50 s because decoding the ~4.4 MB originals on 2 CPU cores was the bottleneck. The schedule was shortened from 300 to 100 epochs (patience 50 → 20). Later runs used a local RTX 4060 (~26 s/epoch).
- **Out of memory.** The first local run crashed at epoch 31 (system RAM) with 16 data-loader processes; the workers were reduced to 4 + 2.
- **Interrupted runs.** E3_scale stopped at epoch 76; its curve stayed below E1_1 at every matched epoch, so it was not repeated. The Colab copy of seed 2024 was stopped after 3 epochs because that seed had been completed locally.

**Limitations.** Only 43 validation and 43 test images, so the spread between runs is noticeable. Size classes depend on a fixed camera distance and the ImageJ calibration. A single dataset and camera setup were used.